# Adaptive Parallel Connected Components — run on Colab

Runtime → Change runtime type → **GPU** (a T4 is fine), then run the cells top to bottom.

Your results go to `results/`. At the end they are shown next to the published run in `results_rtx3050_laptop/`. Absolute times depend on the GPU, so compare the ordering of the versions, not the milliseconds.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2

In [ ]:
!git clone https://github.com/md25052007/gpu-cc-ablation.git /content/gpu-cc-ablation
%cd /content/gpu-cc-ablation

## 1. Build and correctness tests
`make tests` generates 12 small test graphs with awkward shapes and checks every version against the CPU reference. **Do not benchmark anything until it prints ALL TESTS PASSED.**

In [ ]:
!make ARCH=sm_75 && make counters ARCH=sm_75   # sm_75 = T4; use sm_80 on an A100
!make tests

## 2. Get the graphs
Start with the two small ones to check the pipeline end to end, then fetch the rest. Converting `com-Orkut` needs about 6 GB of RAM. If a download fails, the script prints the page to download from by hand (see `REPRODUCE.md`).

In [ ]:
!python3 tools/download.py --list
!python3 tools/download.py web-Google rgg_n_2_21_s0

In [ ]:
!python3 tools/download.py   # the remaining core graphs (takes a while)

## 3. Quick single-graph run

In [ ]:
!./bin/cc data/web-Google.egr --runs 5

## 4. Full experiment
Writes `results/timing.csv`, `counters.csv`, `segments.csv` and `stats.csv`, then the segment sweep and the threshold sweep.

In [ ]:
!python3 tools/bench.py data/*.egr --modes stats,main,counters --runs 5 --tag Colab

In [ ]:
!python3 tools/bench.py data/*.egr --modes sweep-s,sweep-t --runs 3 --tag Colab

## 5. Figures and tables

In [ ]:
!python3 tools/plots.py
import pandas as pd
from IPython.display import Image, display
print('your run:')
display(pd.read_csv('results/figures/table_results.csv'))
print('published run (RTX 3050 Laptop GPU):')
display(pd.read_csv('results_rtx3050_laptop/figures/table_results.csv'))
for f in ['fig1_speedup', 'fig2_breakdown', 'fig3_sweep_s', 'fig4_counters', 'fig5_skew', 'fig6_thresholds']:
    display(Image(f'results/figures/{f}.png'))

## 6. Save your results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/gpu-cc-ablation-results
!cp -r results/* /content/drive/MyDrive/gpu-cc-ablation-results/